# Phần 1. NumPy trong workflow ML/DL

Các bài dưới đây dùng dữ liệu nhỏ để mô phỏng preprocessing, inference và xử lý
tensor trong một pipeline thực tế.

In [14]:
STUDENT_NAME = "Phan Đăng Trọng Tín"  # TODO: Họ và tên
STUDENT_ID = "25521866"    # TODO: MSSV

print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")

Student: Phan Đăng Trọng Tín (25521866)


In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.8)

DATA_CANDIDATES = [
    Path("week02/numpy-pandas-eda-hw/data/automobile_raw.csv"),
    Path("data/automobile_raw.csv"),
    Path("../data/automobile_raw.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Không tìm thấy data/automobile_raw.csv")

print("Data path:", DATA_PATH.resolve())

Data path: C:\Users\LENOVO\OneDrive - VNU-HCMUS\mliot-pyml-2026-hw\week02\numpy-pandas-eda-hw\data\automobile_raw.csv


## N1. Stable softmax cho batch logits

Một classifier trả về `logits` có shape `(batch_size, num_classes)`. Tính softmax
theo từng mẫu bằng cách trừ giá trị lớn nhất trên mỗi hàng trước khi gọi `np.exp`.
Cách viết này tránh overflow khi logits có giá trị lớn.

**Biến đầu ra bắt buộc**

- `shifted_logits`: logits sau khi trừ row-wise maximum.
- `class_probabilities`: xác suất mỗi class, mỗi hàng có tổng bằng 1.
- `predicted_classes`: class có xác suất lớn nhất của từng mẫu.
- `confidence_scores`: xác suất lớn nhất của từng mẫu.

In [16]:
logits = np.array([
    [2.0, 1.0, 0.1],
    [1000.0, 1001.0, 999.0],
    [-2.0, -1.0, 3.0],
    [0.5, 0.5, 0.5],
], dtype=np.float64)

In [17]:
# TODO N1
shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
exp_logits = np.exp(shifted_logits)
class_probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
predicted_classes = np.argmax(class_probabilities, axis=1)
confidence_scores = np.max(class_probabilities, axis=1)

In [18]:
required = [
    "shifted_logits",
    "class_probabilities",
    "predicted_classes",
    "confidence_scores",
]
if not all(name in globals() for name in required):
    print("Complete N1 to run this self-check.")
else:
    assert class_probabilities.shape == logits.shape
    assert np.all(np.isfinite(class_probabilities))
    assert np.allclose(class_probabilities.sum(axis=1), 1.0)
    assert predicted_classes.shape == (logits.shape[0],)
    assert confidence_scores.shape == (logits.shape[0],)
    print("N1 self-check passed")

N1 self-check passed


## N2. Chuẩn hóa train và validation không gây leakage

Mỗi hàng là một mẫu, mỗi cột là một feature. Tính mean/std **chỉ từ `X_train`**,
sau đó dùng cùng thống kê để transform cả train và validation.

**Biến đầu ra bắt buộc**

- `train_feature_mean`, `train_feature_std`: shape `(4,)`.
- `X_train_scaled`: train set đã chuẩn hóa.
- `X_val_scaled`: validation set dùng thống kê từ train.

In [19]:
# Features: height_cm, weight_kg, activity_hours, age
X_train = np.array([
    [170.0, 65.0, 1.2, 22.0],
    [180.0, 80.0, 2.4, 35.0],
    [160.0, 50.0, 0.8, 19.0],
    [175.0, 70.0, 1.5, 28.0],
    [168.0, 60.0, 1.0, 24.0],
    [182.0, 90.0, 3.0, 41.0],
])

X_val = np.array([
    [172.0, 68.0, 1.4, 26.0],
    [190.0, 95.0, 3.4, 45.0],
])

In [20]:
# TODO N2
train_feature_mean = np.mean(X_train, axis=0)
train_feature_std = np.std(X_train, axis=0)

X_train_scaled = (X_train - train_feature_mean) / train_feature_std
X_val_scaled = (X_val - train_feature_mean) / train_feature_std

In [21]:
required = [
    "train_feature_mean",
    "train_feature_std",
    "X_train_scaled",
    "X_val_scaled",
]
if not all(name in globals() for name in required):
    print("Complete N2 to run this self-check.")
else:
    assert X_train_scaled.shape == X_train.shape
    assert X_val_scaled.shape == X_val.shape
    assert np.allclose(X_train_scaled.mean(axis=0), 0.0)
    assert np.allclose(X_train_scaled.std(axis=0), 1.0)
    print("N2 self-check passed")

N2 self-check passed


## N3. Tạo review queue sau inference

Giả sử `class_probabilities` đến từ N1. Một prediction cần được kiểm tra thủ công
nếu dự đoán sai **hoặc** confidence nhỏ hơn `0.70`.

**Biến đầu ra bắt buộc**

- `correct_mask`
- `high_confidence_mask`
- `review_mask`
- `review_indices`

In [22]:
true_labels = np.array([0, 2, 2, 1])
confidence_threshold = 0.70

In [23]:
# TODO N3
correct_mask = (predicted_classes == true_labels)
high_confidence_mask = (confidence_scores >= confidence_threshold)
review_mask = ~(correct_mask & high_confidence_mask) 
review_indices = np.where(review_mask)[0].

SyntaxError: invalid syntax (2895377254.py, line 5)

## N4. Tiền xử lý và augment một batch ảnh

`image_batch_uint8` có layout `(B, H, W, C)`. Chuyển batch về `float32` trong đoạn
`[0, 1]`, sau đó tạo một batch mới được flip ngang. Batch augment phải có bộ nhớ
độc lập để việc chỉnh sửa không làm thay đổi batch đã normalize.

Sau khi tạo batch augment, đặt pixel `augmented_batch[0, 0, 0, 0] = 1.0`.

**Biến đầu ra bắt buộc:** `normalized_batch`, `augmented_batch`.

In [ ]:
image_batch_uint8 = (
    np.arange(2 * 4 * 4 * 3, dtype=np.uint8)
    .reshape(2, 4, 4, 3)
)

In [ ]:
# TODO N4
normalized_batch = image_batch_uint8.astype(np.float32) / 255.0
augmented_batch = np.flip(normalized_batch.copy(), axis=2)
augmented_batch[0, 0, 0, 0] = 1.0

# Phần 2. EDA với Automobile

Đọc `data/data_dictionary.md` trước khi xử lý.

## Câu hỏi mở đầu

1. Mỗi dòng đại diện cho đối tượng gì?
2. Ký hiệu missing value trong CSV là gì?
3. `symboling` có ý nghĩa gì?

**Trả lời**

<!-- Viết câu trả lời tại đây. -->

## D1. Load và inspect raw CSV

Load dữ liệu sao cho dấu `?` vẫn là chuỗi để quan sát ảnh hưởng tới dtype.

**Biến đầu ra bắt buộc**

- `raw_df`: DataFrame raw.
- `raw_shape`: tuple.
- `raw_missing_marker_count`: tổng số dấu `?`.

In [ ]:
# TODO D1
raw_df = pd.read_csv(DATA_PATH)
raw_shape = raw_df.shape
raw_missing_marker_count = (raw_df == '?').sum().sum()

## D2. Missing values và dtype

1. Thay `?` bằng `np.nan`.
2. Chuyển các cột trong `NUMERIC_COLUMNS` bằng `pd.to_numeric`.
3. Tạo báo cáo missing.

**Biến đầu ra bắt buộc:** `df_clean`, `missing_by_column`.

In [ ]:
NUMERIC_COLUMNS = ['symboling', 'normalized_losses', 'wheel_base', 'length', 'width', 'height', 'curb_weight', 'engine_size', 'bore', 'stroke', 'compression_ratio', 'horsepower', 'peak_rpm', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D2
df_clean = raw_df.replace('?', np.nan)
for column in NUMERIC_COLUMNS:
    df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')
missing_by_column = df_clean.isna().sum()

### Giải thích cách làm sạch dữ liệu

- Vì sao không nên fill tất cả numeric columns bằng cùng một giá trị?
- Với `price`, lựa chọn drop hay fill phù hợp hơn cho bài EDA này? Vì sao?
- `normalized_losses` thiếu nhiều dữ liệu hơn các cột khác. Điều này ảnh hưởng thế nào?

**Nhận xét**

<!-- Viết 3--6 câu tại đây. -->

## D3. DataFrame sang NumPy

Dùng sáu cột trong `AUTO_FEATURES`. Drop các dòng thiếu ít nhất một trong
sáu cột, sau đó chuyển sang `float64` NumPy array và chuẩn hóa theo feature.

**Biến đầu ra bắt buộc**

- `analysis_df`
- `X_auto`
- `auto_feature_mean`
- `auto_feature_std`
- `X_auto_scaled`

In [ ]:
AUTO_FEATURES = ['curb_weight', 'engine_size', 'horsepower', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D3
analysis_df = df_clean.dropna(subset=AUTO_FEATURES)
X_auto = analysis_df[AUTO_FEATURES].to_numpy(dtype=np.float64)
auto_feature_mean = np.mean(X_auto, axis=0)
auto_feature_std = np.std(X_auto, axis=0)
X_auto_scaled = (X_auto - auto_feature_mean) / auto_feature_std

## D4. Outlier theo price z-score

Tính z-score của `price` bằng NumPy. Một dòng được xem là outlier trong bài
này khi `abs(z) > 2`.

**Biến đầu ra bắt buộc:** `price_z`, `price_outlier_mask`, `price_outliers`.

In [ ]:
# TODO D4
price_index = AUTO_FEATURES.index('price')
price_z = X_auto_scaled[:, price_index]
price_outlier_mask = np.abs(price_z) > 2
price_outliers = analysis_df[price_outlier_mask]

## D5. Correlation và GroupBy

**Biến đầu ra bắt buộc**

- `engine_price_corr`: Pearson correlation tính bằng NumPy.
- `price_by_body_style`: Series mean price theo `body_style`, sort index.

In [ ]:
# TODO D5
engine_index = AUTO_FEATURES.index('engine_size')
engine_price_corr = np.corrcoef(X_auto[:, engine_index], X_auto[:, price_index])[0, 1]
price_by_body_style = analysis_df.groupby('body_style')['price'].mean().sort_index()

# Phần 3. Visualization và insight

Mỗi biểu đồ cần:

1. một câu hỏi;
2. title, axis labels và unit;
3. lựa chọn chart phù hợp;
4. 1--2 câu nhận xét ngay dưới chart.

## M2.1 Price phân phối như thế nào?

In [ ]:
# TODO M2.1: histogram/KDE của price
plt.figure(figsize=(10, 5))
sns.histplot(data=analysis_df, x='price', kde=True, bins=20)
plt.title('Phân phối của Giá xe (Price)')
plt.xlabel('Giá xe (USD)')
plt.ylabel('Số lượng')
plt.show()

**Nhận xét:** <!-- 1--2 câu --> Giá xe phân phối lệch phải (right-skewed). Đa số các mẫu xe tập trung ở phân khúc giá rẻ và tầm trung từ 5,000 USD đến 15,000 USD; số lượng xe hạng sang (trên 30,000 USD) khá hiếm.

## M2.2 Dataset có cân bằng theo body style không?

In [ ]:
# TODO M2.2: countplot của body_style
plt.figure(figsize=(10, 5))
sns.countplot(data=analysis_df, x='body_style', order=analysis_df['body_style'].value_counts().index)
plt.title('Số lượng xe theo Kiểu dáng (Body Style)')
plt.xlabel('Kiểu dáng xe')
plt.ylabel('Số lượng')
plt.show()

**Nhận xét:** <!-- 1--2 câu --> Dataset không cân bằng. Dòng xe sedan và hatchback chiếm ưu thế vượt trội, trong khi hardtop và convertible có rất ít mẫu quan sát.

## M2.3 Price khác nhau theo body style ra sao?

In [ ]:
# TODO M2.3: boxplot price theo body_style
plt.figure(figsize=(10, 5))
sns.boxplot(data=analysis_df, x='body_style', y='price')
plt.title('Phân bố Giá xe theo Kiểu dáng')
plt.xlabel('Kiểu dáng xe')
plt.ylabel('Giá xe (USD)')
plt.show()

**Nhận xét:** <!-- 1--2 câu --> Xe hardtop và convertible có mức giá trung bình cao nhất, với khoảng biến thiên giá rộng. Trái lại, xe hatchback chủ yếu nằm ở phân khúc giá thấp, hiếm có mẫu vượt quá 15,000 USD.

## M2.4 Engine size liên quan thế nào tới price?

In [ ]:
# TODO M2.4: scatterplot engine_size và price, hue=fuel_type
plt.figure(figsize=(10, 5))
sns.scatterplot(data=analysis_df, x='engine_size', y='price', hue='fuel_type', alpha=0.8)
plt.title('Tương quan giữa Kích thước Động cơ và Giá xe (theo Loại nhiên liệu)')
plt.xlabel('Kích thước động cơ (Engine Size)')
plt.ylabel('Giá xe (USD)')
plt.show()

**Nhận xét:** <!-- 1--2 câu --> Có mối tương quan tuyến tính thuận và rất mạnh: động cơ càng lớn thì giá xe càng cao. Xe chạy động cơ diesel (màu cam) nhìn chung tập trung ở phân khúc động cơ tầm trung nhưng giá vẫn khá cao tương ứng.

## M2.5 Các feature numeric tương quan ra sao?

In [ ]:
# TODO M2.5: correlation heatmap
plt.figure(figsize=(10, 8))
corr_matrix = analysis_df[AUTO_FEATURES].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Biểu đồ Tương quan (Correlation Heatmap) các Đặc trưng số')
plt.show()

**Nhận xét:** <!-- 1--2 câu --> price có sự tương quan thuận cực kỳ mạnh với engine_size (0.87) và horsepower (0.81). Ngược lại, giá xe tương quan nghịch mạnh với mức độ tiết kiệm nhiên liệu như city_mpg (-0.69).

## M2.6 Biểu đồ tự chọn

Đặt một câu hỏi mới, chọn chart phù hợp và không lặp nguyên năm biểu đồ trên.

In [ ]:
# TODO M2.6: biểu đồ tự chọn
plt.figure(figsize=(10, 5))
sns.violinplot(data=analysis_df, x='drive_wheels', y='price', inner='quartile')
plt.title('Giá xe theo Hệ dẫn động (Drive Wheels)')
plt.xlabel('Hệ dẫn động (fwd: cầu trước, rwd: cầu sau, 4wd: 2 cầu)')
plt.ylabel('Giá xe (USD)')
plt.show()

**Nhận xét:** <!-- 1--2 câu --> Xe dẫn động cầu sau (rwd) thường nằm trong nhóm xe đắt tiền và cao cấp nhất. Xe dẫn động cầu trước (fwd) có xu hướng là những dòng xe phổ thông giá rẻ với mức biến động giá rất hẹp.

# Tổng hợp

Viết:

- 3--5 phát hiện chính có dẫn chứng;
- ít nhất 2 hạn chế của dataset;
- một ví dụ về correlation không đồng nghĩa causation;
- một câu hỏi nên phân tích tiếp.

## Tổng hợp của sinh viên

<!-- Viết khoảng 150--250 từ. -->
3--5 phát hiện chính có dẫn chứng:

* Giá xe bị chi phối mạnh mẽ bởi kích thước động cơ (corr = 0.87) và mã lực (corr = 0.81).

* Dữ liệu có xu hướng nghiêng mạnh về các dòng xe phổ thông: trên 80% là xe sedan và hatchback, giá thường thấp hơn 15,000 USD.

* Có một sự đánh đổi rõ ràng giữa khả năng vận hành (mã lực, khối lượng) và mức tiêu hao nhiên liệu (MPG) - chúng có hệ số tương quan nghịch rất mạnh.

* Hệ dẫn động bánh sau (rwd) là một yếu tố phân loại khá tốt cho phân khúc xe đắt tiền.

Ít nhất 2 hạn chế của dataset:

* Tập dữ liệu quá nhỏ (khoảng 200 mẫu ban đầu), gây khó khăn cho việc kết luận thống kê mạnh hoặc huấn luyện các mô hình Machine Learning phức tạp (dễ overfit).

* Dữ liệu thiếu hụt (missing values) nhiều ở một số trường như normalized_losses, làm giảm khả năng khai thác biến này cho mô hình đánh giá rủi ro.

* Một ví dụ về correlation không đồng nghĩa causation (Tương quan không phải là nhân quả):

* Mặc dù city_mpg (số dặm đi được trên một gallon ở thành phố) có tương quan nghịch với price (xe càng tốn xăng giá càng đắt), nhưng việc "làm cho xe tốn xăng hơn" sẽ không khiến xe bán được giá cao hơn. Cả hai yếu tố này đều bị ảnh hưởng bởi biến ẩn thứ 3 (cấu hình động cơ lớn, xe khối lượng nặng của phân khúc xe sang).

Một câu hỏi nên phân tích tiếp:

* Thương hiệu (make) ảnh hưởng như thế nào đến giá xe nếu chúng ta cố định yếu tố về kích thước động cơ (engine_size)? (Xác định "phụ phí thương hiệu").